# SatQuery Phase 4D — BigEarthNet S2 materialization
CPU-only, independently recoverable canonical transfer. Run after S1 succeeds. It never opens raster pixels.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get('SATQUERY_REPO_URL', 'https://github.com/bishuk-dev/SIH-26167-SATQuery.git')
GIT_REF = os.environ['SATQUERY_GIT_REF']
REPO_DIR = Path('/tmp/SIH-26167-SATQuery')
OUTPUT_DIR = Path('/kaggle/working/satquery-output') / os.environ['SATQUERY_REMOTE_OUTPUT']
DATA_ROOT = Path('/kaggle/working/.satquery-phase4-materialize-s2')
EXPECTED_MANIFEST_SHA256 = '615e30273cce8eaa8b0838c07256714a3c874019f6dccd50570cbf1ec4c20bd6'
EXPECTED_FINAL_BYTES = 3_856_128_477
KAGGLE_OUTPUT_BUDGET_BYTES = 20 * 1024**3
REQUIRED_FREE_BYTES = 8 * 1024**3

assert not REPO_DIR.exists(), f'Refusing non-fresh repository path: {REPO_DIR}'
subprocess.run(['git', 'clone', '--no-checkout', '--filter=blob:none', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', GIT_REF], cwd=REPO_DIR, check=True)
assert subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout == ''
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'zstandard>=0.23,<1'], check=True, env={**os.environ, 'PIP_CACHE_DIR': '/tmp/pip-cache'})

manifest = REPO_DIR / 'experiments/phase4_bigearthnet_multisensor/split_manifest.json'
observed_sha = hashlib.sha256(manifest.read_bytes()).hexdigest()
assert observed_sha == EXPECTED_MANIFEST_SHA256, (EXPECTED_MANIFEST_SHA256, observed_sha)
s1_manifests = []
for candidate in Path('/kaggle/input').rglob('package_manifest.json'):
    payload = json.loads(candidate.read_text())
    package = payload.get('packages', {}).get('s1')
    if payload.get('frozen_split_manifest_sha256') == EXPECTED_MANIFEST_SHA256 and package and package.get('file_count') == 36_002:
        s1_manifests.append((candidate, package))
assert len(s1_manifests) == 1, f'Require exactly one verified S1 notebook output; found {len(s1_manifests)}'
s1_manifest_path, s1_package = s1_manifests[0]
s1_package_path = s1_manifest_path.parent / s1_package['package_file']
assert s1_package_path.stat().st_size == s1_package['package_size_bytes']
s1_digest = hashlib.sha256()
with s1_package_path.open('rb') as handle:
    for chunk in iter(lambda: handle.read(8 * 1024**2), b''):
        s1_digest.update(chunk)
assert s1_digest.hexdigest() == s1_package['package_sha256'], 'Attached S1 package SHA-256 mismatch'
free_bytes = shutil.disk_usage('/kaggle/working').free
print({'manifest_sha256': observed_sha, 'working_free_gib': round(free_bytes / 1024**3, 2), 'output_budget_gib': 20, 'estimated_combined_package_gib': round(EXPECTED_FINAL_BYTES / 1024**3, 2)})
assert EXPECTED_FINAL_BYTES <= KAGGLE_OUTPUT_BUDGET_BYTES, 'Selected package estimate exceeds Kaggle output budget'
assert free_bytes >= REQUIRED_FREE_BYTES, f'Need at least {REQUIRED_FREE_BYTES} free bytes; found {free_bytes}'


In [ ]:
command = [
    sys.executable, '-m', 'ml.evaluation.materialize_phase4_bigearthnet',
    '--confirm-full-stream-transfer', '--modality', 's2',
    '--manifest', str(manifest), '--output-root', str(DATA_ROOT),
    '--report', str(OUTPUT_DIR / 'materialization_report.json'),
    '--package-output-dir', str(OUTPUT_DIR), '--delete-loose-after-package',
]
subprocess.run(command, cwd=REPO_DIR, check=True)
assert (OUTPUT_DIR / 'phase4_s2_selected.tar.zst').is_file()
assert (OUTPUT_DIR / 'package_manifest.json').is_file()
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)


In [ ]:
print('Final S2 outputs:')
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file(): print(path.name, path.stat().st_size)
